# 4-2. 瑞利/莱斯衰落信道

**Rayleigh and Rician Fading Channels**


---


本notebook介绍无线通信中最重要的衰落信道模型：瑞利衰落和莱斯衰落。理解这些统计模型对于设计可靠的无线通信系统至关重要。

## 1. 学习目标

- **理解瑞利衰落**：无直视路径时的信号衰落统计特性
- **掌握莱斯衰落**：存在直视分量时的信道建模
- **理解信道参数**：K因子、均方根时延扩展、相干带宽
- **掌握信道仿真**：如何生成符合统计特性的随机信道
- **理解分集技术**：如何对抗衰落提高系统可靠性

## 2. 概念讲解

### 2.1 瑞利衰落信道

当发射端与接收端之间不存在直视路径（NLOS）时，多径效应导致接收信号幅度服从瑞利分布：

$$f(r) = \frac{r}{\sigma^2} e^{-r^2/(2\sigma^2)}, \quad r \geq 0$$

其中σ是标准差。


**瑞利衰落的特点**：
- 信号包络服从瑞利分布
- 相位服从均匀分布
- 适用于密集散射环境

### 2.2 莱斯衰落信道

当存在直视路径（LOS）时，信号包络服从莱斯分布：

$$f(r) = \frac{r}{\sigma^2} e^{-(r^2 + A^2)/(2\sigma^2)} I_0\left(\frac{Ar}{\sigma^2}\right), \quad r \geq 0$$


其中A是直视分量的幅度，I₀是修正贝塞尔函数。

**K因子**：K = A²/(2σ²)，表示直视分量与散射分量的功率比。
- K = 0：瑞利衰落
- K → ∞：高斯信道（无衰落）

## 3. Python演示

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("瑞利/莱斯衰落信道演示环境已准备就绪")

### 3.1 瑞利衰落信道生成

In [ ]:
# 瑞利衰落信道生成
def generate_rayleigh_channel(num_samples, dopplerSpread=10, Fs=1000):
    """
    使用Jakes模型生成瑞利衰落信道
    
    参数:
    - num_samples: 采样点数量
    - dopplerSpread: 多普勒扩展 (Hz)
    - Fs: 采样频率
    
    返回:
    - rayleigh_channel: 瑞利衰落信道序列
    """
    # Jakes模型参数
    N = 8  # 路径数
    wm = 2 * np.pi * dopplerSpread  # 最大多普勒角频率
    
    # 生成N个独立路径
    phases = np.random.uniform(0, 2*np.pi, N)
    fds = np.linspace(-wm, wm, N)
    
    # 时变信道
    t = np.arange(num_samples) / Fs
    channel = np.zeros(num_samples, dtype=complex)
    
    for i in range(N):
        channel += np.exp(1j * (2 * np.pi * fds[i] * t + phases[i]))
    
    # 归一化
    channel = channel / np.sqrt(N)
    
    return channel

def demo_rayleigh_channel():
    """
    演示瑞利衰落信道的特性
    """
    np.random.seed(42)
    
    # 生成信道
    Fs = 1000  # 采样频率 1kHz
    duration = 10  # 信号持续时间 10秒
    num_samples = Fs * duration
    doppler = 10  # 多普勒扩展 10Hz
    
    channel = generate_rayleigh_channel(num_samples, doppler, Fs)
    
    # 计算包络
    envelope = np.abs(channel)
    power_db = 20 * np.log10(envelope + 1e-10)
    
    # 理论瑞利分布
    sigma = 1 / np.sqrt(2)  # 归一化瑞利参数
    r_theory = np.linspace(0, 4, 1000)
    pdf_theory = r_theory / sigma**2 * np.exp(-r_theory**2 / (2*sigma**2))
    
    # 可视化
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 时变信道响应
    ax = axes[0, 0]
    t = np.arange(num_samples) / Fs
    ax.plot(t, envelope, 'b-', linewidth=0.5, alpha=0.7)
    ax.set_xlabel('时间 (s)', fontsize=11)
    ax.set_ylabel('包络', fontsize=11)
    ax.set_title('瑞利衰落信道包络随时间变化', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # 包络分布直方图
    ax = axes[0, 1]
    ax.hist(envelope, bins=100, density=True, alpha=0.7, color='steelblue', 
           edgecolor='white', label='仿真')
    ax.plot(r_theory, pdf_theory, 'r-', linewidth=2, label='理论瑞利分布')
    ax.set_xlabel('包络 r', fontsize=11)
    ax.set_ylabel('概率密度', fontsize=11)
    ax.set_title('瑞利衰落包络分布', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 功率分布（dB）
    ax = axes[1, 0]
    ax.hist(power_db, bins=100, density=True, alpha=0.7, color='green')
    ax.set_xlabel('功率 (dB)', fontsize=11)
    ax.set_ylabel('概率密度', fontsize=11)
    ax.set_title('瑞利衰落功率分布', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # 电平通过率（LCR）和平均电平持续时间（ALD）
    # 固定门限电平
    threshold = 0.5
    crossings = np.sum(np.diff(envelope > threshold) != 0)
    lcr = crossings / duration  # 电平通过率
    avg_duration = np.mean(envelope > threshold) * duration / (crossings + 1e-10)  # 平均持续时间
    
    ax = axes[1, 1]
    thresholds = np.linspace(0.1, 2, 20)
    lcrs = []
    for th in thresholds:
        crossings_th = np.sum(np.diff(envelope > th) != 0)
        lcrs.append(crossings_th / duration)
    
    ax.plot(thresholds, lcrs, 'b-o', linewidth=2, markersize=4)
    ax.set_xlabel('门限电平', fontsize=11)
    ax.set_ylabel('电平通过率', fontsize=11)
    ax.set_title('电平通过率 vs 门限电平', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"瑞利衰落信道参数:")
    print(f"  多普勒扩展: {doppler} Hz")
    print(f"  相干时间 Tc ≈ 0.423/{doppler} = {0.423/doppler:.3f} s")
    print(f"  包络均值: {np.mean(envelope):.4f}")
    print(f"  包络标准差: {np.std(envelope):.4f}")
    print(f"  理论均值: {np.mean(r_theory):.4f}")
    
    return channel, envelope

channel, envelope = demo_rayleigh_channel()

### 3.2 莱斯衰落信道生成

In [ ]:
# 莱斯衰落信道生成
def generate_rician_channel(num_samples, K, dopplerSpread=10, Fs=1000):
    """
    生成莱斯衰落信道
    
    参数:
    - num_samples: 采样点数量
    - K: 莱斯K因子（直射分量与散射分量功率比）
    - dopplerSpread: 多普勒扩展 (Hz)
    - Fs: 采样频率
    
    返回:
    - rician_channel: 莱斯衰落信道序列
    """
    # 归一化因子
    omega = 1  # 总功率
    sigma = np.sqrt(omega / (2 * (K + 1)))  # 散射分量标准差
    A = np.sqrt(K * 2 * sigma**2)  # 直流分量幅度
    
    # 生成散射分量（瑞利部分）
    scattering = generate_rayleigh_channel(num_samples, dopplerSpread, Fs) * sigma * np.sqrt(2)
    
    # 直流分量
    dc = A * np.exp(1j * 0)  # 假设初始相位为0
    
    # 莱斯信道 = 直流分量 + 散射分量
    channel = dc + scattering
    
    return channel, A, sigma

def demo_rician_channel():
    """
    演示不同K值的莱斯衰落信道
    """
    np.random.seed(123)
    
    Fs = 1000
    duration = 5
    num_samples = Fs * duration
    doppler = 10
    
    # 不同K值
    K_values = [0, 3, 7, 15]  # K=0即为瑞利
    colors = ['blue', 'green', 'orange', 'red']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 各K值的信道包络
    for idx, K in enumerate(K_values):
        channel, A, sigma = generate_rician_channel(num_samples, K, doppler, Fs)
        envelope = np.abs(channel)
        
        t = np.arange(num_samples) / Fs
        
        ax = axes[0, idx % 2]
        if idx >= 2:
            ax = axes[1, idx % 2]
        ax.plot(t, envelope, color=colors[idx], linewidth=0.5, alpha=0.7, 
                label=f'K={K}')
        ax.axhline(y=A, color='gray', linestyle='--', alpha=0.7, label=f'A={A:.2f}')
        ax.set_xlabel('时间 (s)', fontsize=11)
        ax.set_ylabel('包络', fontsize=11)
        ax.set_title(f'莱斯衰落 K={K}', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # K值对包络分布的影响
    ax = axes[1, 0] if K_values[0] != 0 else axes[0, 0]
    for idx, K in enumerate(K_values):
        channel, A, sigma = generate_rician_channel(10000, K, doppler, Fs)
        envelope = np.abs(channel)
        ax.hist(envelope, bins=100, density=True, alpha=0.5, 
               color=colors[idx], label=f'K={K}')
    
    ax.set_xlabel('包络 r', fontsize=11)
    ax.set_ylabel('概率密度', fontsize=11)
    ax.set_title('不同K值的包络分布', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n莱斯衰落信道参数:")
    print(f"  K因子: 直射分量与散射分量功率比")
    print(f"  K=0: 瑞利衰落（无直视路径）")
    print(f"  K→∞: 近高斯信道（直射路径主导）")
    print(f"  K=3-7: 典型微蜂窝场景")
    print(f"  K>15: 典型宏蜂窝/视距场景")
    
    return channel, envelope

channel, envelope = demo_rician_channel()

### 3.3 衰落信道对通信系统的影响

In [ ]:
# 衰落信道对通信系统的影响
def demo_fading_effect():
    """
    演示瑞利衰落对误码率的影响
    """
    np.random.seed(456)
    
    # 仿真参数
    EbN0_db = np.arange(0, 20, 2)  # dB
    EbN0 = 10 ** (EbN0_db / 10)
    
    # 理论误码率（ BPSK, 瑞利衰落）
    # 在瑞利衰落信道中，误码率需要信道估计或分集来改善
    
    # 理论AWGN误码率
    Pb_awgn = 0.5 * np.exp(-EbN0)
    
    # 无分集时的瑞利衰落误码率（需要完美信道估计）
    # Pb_rayleigh = 0.5 * (1 - np.sqrt(EbN0 / (1 + EbN0)))
    
    # 仿真：使用随机信道
    num_bits = 10000
    ber_sim = []
    
    for snr_db in EbN0_db:
        # 生成瑞利衰落信道
        channel = generate_rayleigh_channel(num_bits, dopplerSpread=10, Fs=1000)
        
        # BPSK调制
        bits = np.random.randint(0, 2, num_bits)
        symbols = 2 * bits - 1
        
        # 通过信道（归一化信道增益）
        received = channel * symbols / np.sqrt(np.mean(np.abs(channel)**2))
        
        # 添加噪声
        noise_var = 1 / EbN0[EbN0_db == snr_db][0]
        noise = np.sqrt(noise_var/2) * (np.random.randn(num_bits) + 1j * np.random.randn(num_bits))
        received = received + noise
        
        # 检测（假设完美信道估计）
        detected_bits = np.real(channel.conj() * received) > 0
        errors = np.sum(bits != detected_bits)
        ber_sim.append(errors / num_bits)
    
    ber_sim = np.array(ber_sim)
    
    # 可视化
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.semilogy(EbN0_db, Pb_awgn, 'b-o', linewidth=2, markersize=6, 
                 label='AWGN信道（理论）')
    ax.semilogy(EbN0_db, ber_sim, 'r-s', linewidth=2, markersize=6, 
                 label='瑞利衰落信道（仿真）')
    
    ax.set_xlabel('Eb/N0 (dB)', fontsize=11)
    ax.set_ylabel('误码率 (BER)', fontsize=11)
    ax.set_title('BPSK在不同信道下的误码率性能', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3, which='both')
    ax.set_ylim(1e-4, 1)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n瑞利衰落的影响:")
    print(f"  在Eb/N0=10dB时:")
    print(f"    AWGN理论BER: {Pb_awgn[EbN0_db == 10][0]:.4f}")
    print(f"    瑞利衰落仿真BER: {ber_sim[EbN0_db == 10][0]:.4f}")
    print(f"  衰落导致约10-100倍的误码率增加")
    
    return EbN0_db, ber_sim

EbN0_db, ber_sim = demo_fading_effect()

## 4. 思考题

1. **瑞利vs莱斯**：在城市环境中（密集建筑、无直视路径），使用哪种衰落模型更合适？
2. **K因子估计**：如何从实测数据中估计莱斯信道的K因子？
3. **分集技术**：为什么分集技术能有效对抗衰落？计算2分支分集的性能增益。
4. **多普勒扩展**：高速移动（如300km/h）会对信道产生什么影响？如何应对？

---

## 总结

本notebook介绍了无线衰落信道的核心模型：
- **瑞利衰落**：无直视路径时，包络服从瑞利分布，适用于NLOS场景
- **莱斯衰落**：存在直视路径时，包络服从莱斯分布，K因子描述LOS强度
- **信道参数**：相干时间、均方根时延扩展等描述信道时间色散特性
- **衰落影响**：显著恶化系统性能，需要分集等技术对抗